# 4章 サンプル（numpy版） ― 「|」「-」「/」の3クラス判定

numpy3.pyのColab版。04-2Train3.ipynb（for文だけで書いた版）とまったく同じデータ・
まったく同じ計算を、numpyの行列演算（`@`）でまとめて書いている。

data3フォルダの15ファイルを読み込んでいたオリジナル版から、Colabでもそのまま動くように、
データをプログラム内のリテラルに変えてある。


In [ ]:
import numpy as np

# 04-2Train3.ipynbと完全に同じ15件のデータ
# (ファイル名の先頭の数字が正解クラス： 0="|", 1="-", 2="/")
RAW_DATA = [
    ("0_01.txt", [
        "................",
        "................",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "................",
        "................",
    ]),
    ("0_02.txt", [
        "................",
        "................",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "................",
        "................",
    ]),
    ("0_03.txt", [
        "................",
        "................",
        "................",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "................",
        "................",
        "................",
    ]),
    ("0_04.txt", [
        "................",
        "................",
        "................",
        ".........■......",
        ".........■......",
        ".........■......",
        ".........■......",
        ".........■......",
        ".........■......",
        ".........■......",
        ".........■......",
        ".........■......",
        ".........■......",
        "................",
        "................",
        "................",
    ]),
    ("0_05.txt", [
        "................",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "................",
    ]),
    ("1_01.txt", [
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "..■■■■■■■■■■■■..",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("1_02.txt", [
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "..■■■■■■■■■■■■..",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("1_03.txt", [
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "...■■■■■■■■■■...",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("1_04.txt", [
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "...■■■■■■■■■■...",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("1_05.txt", [
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        ".■■■■■■■■■■■■■■.",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("2_01.txt", [
        "................",
        "................",
        ".............■..",
        "............■...",
        "...........■....",
        "..........■.....",
        ".........■......",
        "........■.......",
        ".......■........",
        "......■.........",
        ".....■..........",
        "....■...........",
        "...■............",
        "..■.............",
        "................",
        "................",
    ]),
    ("2_02.txt", [
        "................",
        "................",
        "............■...",
        "...........■....",
        "..........■.....",
        ".........■......",
        "........■.......",
        ".......■........",
        "......■.........",
        ".....■..........",
        "....■...........",
        "...■............",
        "..■.............",
        "................",
        "................",
        "................",
    ]),
    ("2_03.txt", [
        "................",
        "..............■.",
        ".............■..",
        "............■...",
        "...........■....",
        "..........■.....",
        ".........■......",
        "........■.......",
        ".......■........",
        "......■.........",
        ".....■..........",
        "....■...........",
        "...■............",
        "..■.............",
        "................",
        "................",
    ]),
    ("2_04.txt", [
        "................",
        "................",
        "................",
        ".............■..",
        "............■...",
        "...........■....",
        "..........■.....",
        ".........■......",
        "........■.......",
        ".......■........",
        "......■.........",
        ".....■..........",
        "....■...........",
        "...■............",
        "................",
        "................",
    ]),
    ("2_05.txt", [
        "................",
        "................",
        "..............■.",
        ".............■..",
        "............■...",
        "...........■....",
        "..........■.....",
        ".........■......",
        "........■.......",
        ".......■........",
        "......■.........",
        ".....■..........",
        "....■...........",
        "...■............",
        "................",
        "................",
    ]),
]


## データを読み込む

In [ ]:
class TrainingData:
    def __init__(self, filename, rows):
        self.filename = filename
        self.class_id = int(filename[0])
        self.bits = [1.0 if ch == "■" else 0.0 for row in rows for ch in row]


training_data = [TrainingData(name, rows) for name, rows in RAW_DATA]

# ------------------------------------
# X 15ファイル分のビット構成を作る
# 15件 × 256入力
# ------------------------------------
X = np.array([td.bits for td in training_data])

print("X.shape =", X.shape)

# ------------------------------------
# target を作る　各ファイルの正解位置に1.0をセット
# 15件 × 3クラス
# ------------------------------------
targets = np.zeros((len(training_data), 3))

for i, td in enumerate(training_data):
    targets[i][td.class_id] = 1.0

print("targets.shape =", targets.shape)


## モデル（3クラス分の重み・バイアス）

In [ ]:
weights = np.zeros((3, 256))
bias = np.zeros(3)

print("weights.shape =", weights.shape)
print("bias.shape =", bias.shape)


## 学習ループ

04-2Train3.ipynbのfor文がやっていたことを、numpyの行列演算1行で表している。

- `X @ weights.T + bias`：(15×256) @ (256×3) → (15×3) ＝ 全15件・全3クラス分の予測を一度に計算
- `gradient.T @ X`：(3×15) @ (15×256) → (3×256) ＝ 全件分の勾配をweightの形にまとめて計算


In [ ]:
rate = 0.04 / 4
epochs = 600

for epoch in range(epochs):

    # ----------------------------
    # Prediction　(15×256) @ (256×3)　> (15×3)
    # ----------------------------
    prediction = X @ weights.T + bias

    # ----------------------------
    # diff 15x3 - 15x3 = 15x3
    # ----------------------------
    diff = prediction - targets

    # ----------------------------
    # Loss 15x3 = float(単一)
    # ----------------------------
    loss = np.sum(diff ** 2)

    # ----------------------------
    # Prediction に対する gradient
    # d/dPrediction (diff^2) = 2 * diff
    # ----------------------------
    gradient = 2 * diff

    # ----------------------------
    # weight の gradient
    # (3x15) @ (15x256) > 3x256
    # ----------------------------
    weight_gradient = gradient.T @ X

    # ----------------------------
    # bias の gradient　15件分を合計
    # ----------------------------
    bias_gradient = np.sum(gradient, axis=0)

    # ----------------------------
    # weight / bias 更新
    # ----------------------------
    weights -= rate * weight_gradient
    bias -= rate * bias_gradient

    if epoch % 50 == 0:
        print("epoch =", epoch, "loss =", loss)


## 学習結果

In [ ]:
print()
print("----- result -----")

prediction = X @ weights.T + bias
for i, td in enumerate(training_data):
    answer = np.argmax(prediction[i])
    print(td.filename, "target =", td.class_id,
          "prediction =", prediction[i], "answer =", answer)
